In [21]:
# importamos las librerías que necesitamos

# Tratamiento de datos
# -----------------------------------------------------------------------
import pandas as pd
import numpy as np
import warnings #Esto sirve para manejar las advertencias que puedan surgir durante la ejecución del código
warnings.filterwarnings('ignore')

# Imputación de nulos usando métodos avanzados estadísticos
# -----------------------------------------------------------------------
from sklearn.impute import SimpleImputer #para imputar con la media, mediana o moda
from sklearn.experimental import enable_iterative_imputer #para poder usar el IterativeImputer, que es un método de imputación más avanzado que el SimpleImputer
from sklearn.impute import IterativeImputer #para imputar con un modelo de regresión, es decir, que se entrena un modelo para predecir los valores nulos a partir de las demás variables
from sklearn.impute import KNNImputer #para imputar con el método de los k vecinos más cercanos, es decir, que se imputan los valores nulos con la media de los k vecinos más cercanos, donde la cercanía se mide en función de las demás variables

# Librerías de visualización
# -----------------------------------------------------------------------
import seaborn as sns
import matplotlib.pyplot as plt
# Configuración
# -----------------------------------------------------------------------
# Mostrar todas las columnas sin truncar
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)

print("✅ Librerías cargadas correctamente")

✅ Librerías cargadas correctamente


In [22]:
df = pd.read_csv('hr.csv')

print(f"El número de filas y columnas es: {df.shape[0]} filas y {df.shape[1]} columnas")

El número de filas y columnas es: 1474 filas y 35 columnas


In [24]:
df.isnull().sum()

Age                          73
Attrition                     0
BusinessTravel              117
DailyRate                     0
Department                   29
DistanceFromHome              0
Education                     0
EducationField               58
EmployeeCount                 0
EmployeeNumber                0
EnvironmentSatisfaction       0
Gender                        0
HourlyRate                    0
JobInvolvement                0
JobLevel                      0
JobRole                       0
JobSatisfaction              29
MaritalStatus               132
MonthlyIncome                14
MonthlyRate                   0
NumCompaniesWorked            0
Over18                        0
OverTime                     44
PercentSalaryHike             0
PerformanceRating             0
RelationshipSatisfaction      0
StandardHours               164
StockOptionLevel              0
TotalWorkingYears             0
TrainingTimesLastYear        88
WorkLifeBalance               0
YearsAtC

In [5]:
#1 y 2 identificar valores con nulos
nulos = df.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
print(nulos)

StandardHours            164
YearsWithCurrManager     148
MaritalStatus            132
BusinessTravel           117
TrainingTimesLastYear     88
Age                       73
EducationField            58
OverTime                  44
Department                29
JobSatisfaction           29
MonthlyIncome             14
dtype: int64


In [ ]:
## Análisis detallado de valores nulos

#Calculamos exactamente cuántos nulos hay en cada columna y qué porcentaje representan sobre el total. Un porcentaje alto de nulos puede indicar que esa columna no es fiable.

nulos = df.isnull().sum()
pct   = (nulos / len(df) * 100).round(2)

resumen_nulos = pd.DataFrame({
    'Nulos': nulos,
    'Porcentaje (%)': pct
}).sort_values('Nulos', ascending=False)

print("Columnas CON nulos:")
print(resumen_nulos[resumen_nulos['Nulos'] > 0].to_string())
print(f"\nColumnas SIN nulos: {(nulos == 0).sum()}")
print(f"Total de valores nulos en el dataset: {nulos.sum()}")

Columnas CON nulos:
                       Nulos  Porcentaje (%)
StandardHours            164           11.13
YearsWithCurrManager     148           10.04
MaritalStatus            132            8.96
BusinessTravel           117            7.94
TrainingTimesLastYear     88            5.97
Age                       73            4.95
EducationField            58            3.93
OverTime                  44            2.99
JobSatisfaction           29            1.97
Department                29            1.97
MonthlyIncome             14            0.95

Columnas SIN nulos: 24
Total de valores nulos en el dataset: 896


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1474 entries, 0 to 1473
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Age                       1401 non-null   float64
 1   Attrition                 1474 non-null   str    
 2   BusinessTravel            1357 non-null   str    
 3   DailyRate                 1474 non-null   int64  
 4   Department                1445 non-null   str    
 5   DistanceFromHome          1474 non-null   int64  
 6   Education                 1474 non-null   int64  
 7   EducationField            1416 non-null   str    
 8   EmployeeCount             1474 non-null   int64  
 9   EmployeeNumber            1474 non-null   int64  
 10  EnvironmentSatisfaction   1474 non-null   int64  
 11  Gender                    1474 non-null   str    
 12  HourlyRate                1474 non-null   int64  
 13  JobInvolvement            1474 non-null   int64  
 14  JobLevel           

In [8]:
columnas_mediana = ["Age", "YearsWithCurrManager", "TrainingTimesLastYear", "JobSatisfaction","MonthlyIncome"]
 
for col in columnas_mediana:
    df[col] = df[col].fillna(df[col].median())
 

In [17]:
#eliminamos la columna StandardHours porque tiene el mismo valor en todas las filas, por lo que no aporta información útil para el análisis y puede generar ruido en los modelos de machine learning.
df = df.drop('StandardHours', axis=1)

In [36]:
columnas_categoricas = ['MaritalStatus','Department', 'BusinessTravel', 'EducationField', 'OverTime']

def imputar_categoricas(df, columnas):
    for col in columnas:
        nulos = df[col].isnull().sum()
        total = len(df)
        porcentaje_nulos = nulos / total

        if porcentaje_nulos > 0.20:
            # Si hay más del 20% de nulos → rellenar con "Desconocido"
            df[col] = df[col].fillna("Desconocido")
            print(f"{col} se va a imputar con Desconocido")
        else:
            # Menos del 20% de nulos
            valores = df[col].value_counts(dropna=True)
            if len(valores) > 0:
                # Calcular diferencia porcentual entre los dos más frecuentes
                primero = valores.iloc[0]
                proporcion = primero / total #es decir coge las veces que aparece y la divide entre el total de datos
                if proporcion > 0.50:
                    moda = df[col].mode()[0]
                    df[col] = df[col].fillna(moda)
                    print(f"{col} se va a imputar con la moda")
                else:
                    # Si no → usar "Desconocido"
                    df[col] = df[col].fillna("Desconocido")
                    print(f"{col} se va a imputar con Desconocido")
            else:
                print("Error.")
    return df

In [37]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1474 entries, 0 to 1473
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Age                       1401 non-null   float64
 1   Attrition                 1474 non-null   str    
 2   BusinessTravel            1474 non-null   str    
 3   DailyRate                 1474 non-null   int64  
 4   Department                1474 non-null   str    
 5   DistanceFromHome          1474 non-null   int64  
 6   Education                 1474 non-null   int64  
 7   EducationField            1474 non-null   str    
 8   EmployeeCount             1474 non-null   int64  
 9   EmployeeNumber            1474 non-null   int64  
 10  EnvironmentSatisfaction   1474 non-null   int64  
 11  Gender                    1474 non-null   str    
 12  HourlyRate                1474 non-null   int64  
 13  JobInvolvement            1474 non-null   int64  
 14  JobLevel           

In [39]:
imputar_categoricas(df, columnas_categoricas)

MaritalStatus se va a imputar con Desconocido
Department se va a imputar con la moda
BusinessTravel se va a imputar con la moda
EducationField se va a imputar con Desconocido
OverTime se va a imputar con la moda


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41.0,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,sALES eXECUTIVE,4.0,Single,5993.0,19479,8,Y,Yes,11,3,1,80.0,0,8,0.0,1,6,4,0,5.0
1,49.0,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,rESEARCH sCIENTIST,2.0,Married,5130.0,24907,1,Y,No,23,4,4,NaN,1,10,3.0,3,10,7,1,7.0
2,37.0,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,lABORATORY tECHNICIAN,3.0,Single,2090.0,2396,6,Y,Yes,15,3,2,NaN,0,7,3.0,3,0,0,0,0.0
3,33.0,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,rESEARCH sCIENTIST,3.0,Married,2909.0,23159,1,Y,Yes,11,3,3,80.0,0,8,3.0,3,8,7,3,0.0
4,27.0,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,lABORATORY tECHNICIAN,2.0,Married,3468.0,16632,9,Y,No,12,3,4,80.0,1,6,3.0,3,2,2,2,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1469,34.0,No,Travel_Rarely,628,Research & Development,8,3,Medical,1,2068,2,Male,82,4,2,lABORATORY tECHNICIAN,3.0,Married,4404.0,10228,2,Y,No,12,3,1,NaN,0,6,3.0,4,4,3,1,2.0
1470,28.0,No,Travel_Rarely,866,Sales,5,3,Medical,1,1469,4,Male,84,3,2,sALES eXECUTIVE,1.0,Single,8463.0,23490,0,Y,No,18,3,4,NaN,0,6,4.0,3,5,4,1,NaN
1471,53.0,No,Travel_Rarely,1084,Research & Development,13,2,Medical,1,250,4,Female,57,4,2,mANUFACTURING dIRECTOR,1.0,Divorced,4450.0,26250,1,Y,No,11,3,3,NaN,2,5,3.0,3,4,2,1,3.0
1472,24.0,Yes,Travel_Rarely,240,Human Resources,22,1,Human Resources,1,1714,4,Male,58,1,1,hUMAN rESOURCES,3.0,Married,1555.0,11585,1,Y,No,11,3,3,80.0,1,1,2.0,3,1,0,0,0.0
